# Theme 4 — How AI Bug-Fixing Practice Evolves (Dec 2024 – Feb 2026)

Three simple longitudinal questions on the 15-month data:

- **NQ1 — How does each quality metric change over time?** (acceptance, time-to-merge, patch size, revision rate, per agent)
- **NQ2 — Do repositories change which agent they use over time?**
- **NQ3 — Do developers add/modify agent-instruction files over time?**

In [1]:
import sys
sys.path.insert(0, '.')
from analysis_utils import (
    load_fix_prs, load_commits, load_commit_details, build_revision_stats,
    merge_rate, classify_file_role, set_plot_style, save_fig,
    AGENTS, AGENT_COLORS, THEME4_DIR, MIN_N_PROP, MIN_N_MEDIAN,
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
set_plot_style()

In [2]:
# Load data
df        = load_fix_prs()
agents_df = df[df['is_agent'] & df['agent'].isin(AGENTS)].copy()
human_df  = df[~df['is_agent']].copy()
commits   = load_commits()
details   = load_commit_details()
rev_stats = build_revision_stats(df, commits, details)
agent_rev = rev_stats[rev_stats['agent'].isin(AGENTS)].copy()
months    = sorted(df['month'].unique())
idx       = [str(m) for m in months]
print('Months:', idx[0], '->', idx[-1], f'({len(months)} months)')

Loading fix PRs from HuggingFace ...


  AIDev repo coverage: 8,959 distinct repos
  Survivorship cutoff at 2026-01-29: dropped 33,123 recent PRs
  Fix PRs loaded: 371,577  |  Agent: 108,080  |  Human: 263,497


Loading commits from HuggingFace ...


  Commits loaded: 1,156,238
Loading commit details from HuggingFace ...


  Commit details loaded: 7,451,150


Months: 2024-12 -> 2026-01 (14 months)


## NQ1 — How does each metric change over time?
One line per agent. (Months with too few PRs are dropped: n>=30 for rates, n>=20 for medians.)

In [3]:
# Production lines added per PR (for patch size)
details['role'] = details['filename'].map(classify_file_role)
prod = details[details['role'] == 'prod'].groupby('pr_id')['additions'].sum().rename('prod_added')
df_size = df.merge(prod, left_on='id', right_index=True, how='left')

def monthly(agent, metric):
    out = []
    for m in months:
        if metric == 'acceptance':
            s = agents_df[(agents_df.month==m) & (agents_df.agent==agent)]
            out.append(s['is_merged'].mean()*100 if len(s) >= MIN_N_PROP else None)
        elif metric == 'ttm':
            s = agents_df[(agents_df.month==m) & (agents_df.agent==agent) & agents_df.is_merged]
            out.append(s['hours_to_merge'].median() if len(s) >= MIN_N_MEDIAN else None)
        elif metric == 'prodsize':
            s = df_size[(df_size.month==m) & (df_size.agent==agent) & df_size.is_agent]
            out.append(s['prod_added'].median() if len(s) >= MIN_N_MEDIAN else None)
        elif metric == 'revision':
            s = agent_rev[(agent_rev.month==m) & (agent_rev.agent==agent)]
            out.append((s['num_commits']>1).mean()*100 if len(s) >= MIN_N_PROP else None)
    return out

METRICS = [('acceptance','Acceptance %'), ('ttm','Median time-to-merge (h)'),
           ('prodsize','Median production lines'), ('revision','Revision rate %')]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
x = list(range(len(idx)))
for ax,(mk,lbl) in zip(axes.ravel(), METRICS):
    for a in AGENTS:
        ax.plot(idx, np.array([v if v is not None else np.nan for v in monthly(a,mk)], dtype=float),
                'o-', color=AGENT_COLORS[a], label=a, linewidth=1.6, markersize=4)
    ax.axvline('2025-07', color='red', linestyle=':', linewidth=1.2)
    ax.set_title(lbl); ax.set_xticks(x[::2]); ax.set_xticklabels(idx[::2], rotation=45, ha='right', fontsize=8)
axes.ravel()[0].legend(fontsize=8, ncol=2)
fig.suptitle('NQ1: Bug-Fix Quality Metrics Over Time, per Agent', fontsize=13)
fig.tight_layout()
save_fig(fig, 'nq1_metrics_over_time', THEME4_DIR)

  -> Saved: results\theme4_figures\nq1_metrics_over_time.png


WindowsPath('results/theme4_figures/nq1_metrics_over_time.png')

## NQ2 — Do repositories change which agent they use over time?
Agent share of bug-fix PRs each month, plus the share of repos that change their main agent.

In [4]:
# Agent share over time
per_agent = (agents_df.groupby(['month','agent']).size().unstack(fill_value=0)
             .reindex(columns=AGENTS, fill_value=0))
share = per_agent.div(per_agent.sum(axis=1), axis=0) * 100

# Share of repos that change their main (most-used) agent across months
g = agents_df.groupby(['repo','month','agent']).size().reset_index(name='n').sort_values(['repo','month','n'])
dom = g.drop_duplicates(['repo','month'], keep='last')[['repo','month','agent']]
multi = dom.groupby('repo').filter(lambda x: x['month'].nunique() >= 2)
switched = multi.groupby('repo')['agent'].nunique().gt(1).sum()
n_multi = multi['repo'].nunique()
print(f'Repos using agents in >=2 months: {n_multi:,}')
print(f'  changed main agent: {switched:,} ({100*switched/n_multi:.1f}%)')
print(f'  kept same agent   : {n_multi-switched:,} ({100*(n_multi-switched)/n_multi:.1f}%)')

fig, ax = plt.subplots(figsize=(13, 5))
for a in AGENTS:
    ax.plot(idx, share[a].values, 'o-', color=AGENT_COLORS[a], label=a, linewidth=1.8)
ax.axvline('2025-07', color='red', linestyle=':', linewidth=1.2, label='AIDev cutoff')
ax.set_ylabel('Share of agent bug-fix PRs (%)'); ax.set_xlabel('Month')
ax.set_title('NQ2: Agent Share of Bug-Fix PRs Over Time')
plt.xticks(rotation=45, ha='right'); ax.legend()
fig.tight_layout()
save_fig(fig, 'nq2_agent_share_over_time', THEME4_DIR)

Repos using agents in >=2 months: 3,222
  changed main agent: 479 (14.9%)
  kept same agent   : 2,743 (85.1%)


  -> Saved: results\theme4_figures\nq2_agent_share_over_time.png


WindowsPath('results/theme4_figures/nq2_agent_share_over_time.png')

## NQ3 — Do developers add/modify agent-instruction files over time?
Bug-fix PRs that touch `CLAUDE.md`, `.cursorrules`, `copilot-instructions.md`, `AGENTS.md`, etc.

*Limitation: the dataset only has bug-fix PRs, so this is a lower bound — most instruction-file edits happen in other PRs we don't see.*

In [5]:
instr_pat = r'CLAUDE\.md|AGENTS?\.md|GEMINI\.md|\.cursorrules|\.cursor/rules/|copilot-instructions|\.windsurfrules|\.clinerules'
details['is_instr'] = details['filename'].str.contains(instr_pat, case=False, regex=True, na=False)
instr_pr_ids = set(details.loc[details['is_instr'], 'pr_id'])
df['touches_instr'] = df['id'].isin(instr_pr_ids)

monthly_instr = df[df.touches_instr].groupby('month').size().reindex(months, fill_value=0)
cum_repos = (df[df.touches_instr].groupby('repo')['month'].min()
             .value_counts().reindex(months, fill_value=0).cumsum())
print(f'Fix PRs touching an instruction file: {df.touches_instr.sum():,} '
      f'({df.touches_instr.mean()*100:.2f}%), across {df.loc[df.touches_instr,"repo"].nunique():,} repos')

# simple acceptance comparison (plain percentages)
instr_repos = set(df.loc[df.touches_instr, 'repo'])
r_in  = agents_df[agents_df.repo.isin(instr_repos)]['is_merged'].mean()*100
r_out = agents_df[~agents_df.repo.isin(instr_repos)]['is_merged'].mean()*100
print(f'Agent acceptance — repos with instr files: {r_in:.1f}%  |  without: {r_out:.1f}%')

fig, ax = plt.subplots(figsize=(13, 5))
x = list(range(len(idx)))
ax.bar(x, monthly_instr.values, color='#2ca02c', alpha=0.6, label='PRs touching instr files (monthly)')
ax2 = ax.twinx()
ax2.plot(x, cum_repos.values, 'o-', color='#1f77b4', label='cumulative repos')
ax.set_xticks(x[::2]); ax.set_xticklabels(idx[::2], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Fix PRs touching instr files', color='#2ca02c')
ax2.set_ylabel('Cumulative repos', color='#1f77b4')
ax.set_title('NQ3: Agent-Instruction-File Changes Over Time')
fig.tight_layout()
save_fig(fig, 'nq3_instruction_files_over_time', THEME4_DIR)

Fix PRs touching an instruction file: 2,222 (0.60%), across 837 repos
Agent acceptance — repos with instr files: 85.7%  |  without: 82.9%


  -> Saved: results\theme4_figures\nq3_instruction_files_over_time.png


WindowsPath('results/theme4_figures/nq3_instruction_files_over_time.png')